# Анализ выборочных цен ананасов (2021–2025)
Автор: Карасёва Яна  
Цель: исследовать распределение цен ананасов в Москве и выявить статистические особенности данных.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import statistics
import numpy as np
import pandas as pd
from scipy.stats import norm

## Загрузка данных
Используем столбец "Средняя цена" из файла ananas.xlsx.

In [ ]:
url = "https://raw.githubusercontent.com/ya227/pineapple-price-analysis/main/data/ananas.xlsx"
file = pd.read_excel(url, engine="openpyxl")
mas  = file.iloc[1:, 1]
n = len(mas)
sort_mas = np.array(sorted(mas))

## Описательная статистика
Рассчитываем основные метрики: среднее, медиану, моду, дисперсию, стандартное отклонение.

In [ ]:
print('Объем выборки: ', n)
sr = statistics.mean(mas)
print("Среднее арифметическое: ", sr)
dis = statistics.variance(mas)
print("дисперсия: ", dis)
med = statistics.median(mas)
print("медиана: ", med)
stotkl = statistics.pstdev(mas)
print("стандартное отклонение: ", stotkl)
moda = statistics.mode(mas)
print("мода: ", moda)

## Квартильный анализ
Определяем квартильные значения и межквартильный диапазон.

In [ ]:
Q1 = np.quantile(mas, 0.25)
print("Первый квартиль: ", Q1)
Q2 = np.quantile(mas, 0.5)
print("Второй квартиль: ", Q2)
Q3 = np.quantile(mas, 0.75)
print("Третий квартиль: ", Q3)
MKR = Q3 - Q1
print('Межквартильный диапозон: ', MKR )

## Визуализация данных
Строим вариационный ряд, boxplot, гистограммы и эмпирическую функцию распределения.

In [ ]:
plt.title('Вариационный ряд выборочных цен ананасов с 04.06.2021 по 14.03.2025')
plt.xlabel('порядковый номер')
plt.ylabel('Цены')
q = list(range(1,n+1))
plt.scatter(q, sort_mas)
plt.grid()
plt.show()

In [ ]:
left = med - 1.5 * MKR
right = med + 1.5 * MKR
plt.boxplot(sort_mas)
plt.axhline(Q1, color="r", label = 'Нижний квантиль')
plt.axhline(Q3, color="y", label = 'Верхний квантиль')
plt.axhline(med, color="b", label = 'Медиана')
plt.axhline(left, color="g", label = 'Нижний ус')
plt.axhline(right, color="g", label = 'Верхний ус')
plt.legend()
plt.title("Boxplot")
plt.ylabel("Цены ананасов")
plt.grid(True)
plt.show()

In [ ]:
maxnumber = max(sort_mas)
minnumber = min(sort_mas)
column = int(n / 10)
scope = maxnumber - minnumber
step = scope / (column - 1)
border = list()
border.append(-np.inf)
border.append(minnumber + step / 2)
for i in range(column - 2):
    border.append(border[-1] + step)
border.append(np.inf)
kolv = list()
for i in range (len(border) - 1):
    kolv.append(len(sort_mas[np.logical_and(sort_mas>=border[i], sort_mas<border[i+1])]))
ver = list()
for i in range(column):
    ver.append(kolv[i]/n)

fig,ax = plt.subplots()
for k in range(len(border) - 1):
    rect = Rectangle((border[k], 0), width=step, height=kolv[k], edgecolor='r', facecolor='blue', alpha = 0.5)
    ax.add_patch(rect)
rect = Rectangle((border[1], 0), width=-step, height=kolv[0], edgecolor='r', facecolor='blue', alpha = 0.5)
ax.add_patch(rect)
plt.grid(True)
ax.set_xlim(minnumber-2*step, maxnumber+2*step)
ax.set_ylim(0, max(kolv) + 1)
plt.title("Частотная гистограмма")
plt.xlabel("Цены ананасов")
plt.ylabel("Количество")
plt.show()

fig2,ax2 = plt.subplots()
ax2.set_xlim(minnumber-2*step, maxnumber+2*step)
ax2.set_ylim(0, max(ver) + 0.01)
for k in range(len(border) - 1):
    rect = Rectangle((border[k], 0), width=step, height=ver[k], edgecolor='r', facecolor='blue', alpha = 0.5)
    ax2.add_patch(rect)
rect = Rectangle((border[1], 0), width=-step, height=ver[0], edgecolor='r', facecolor='blue', alpha = 0.5)
ax2.add_patch(rect)
plt.grid(True)
plt.title("Вероятностная гистограмма")
plt.xlabel("Цены ананасов")
plt.ylabel("вероятнось попадения в интервал")
plt.axvline(sr, color='black', label='Мат ожидание')
plt.axvline(med, color = 'g', label = 'Медиана')
x = np.linspace(sr - 4*stotkl, sr + 4*stotkl, 1000)
pdf = norm.pdf(x, sr, stotkl)
plt.plot(x, pdf, color = 'orange', label = 'Нормальное распределение')
plt.legend()
plt.show()

In [ ]:
ecdf = 0
plt.hlines(ecdf, 0, sort_mas[0], color = 'r')
for i in range(1, n):
    ecdf += 1 / n
    plt.hlines(ecdf, sort_mas[i-1], sort_mas[i], color = 'r')
    plt.scatter(sort_mas[i-1], ecdf, color = 'white', edgecolors='r')
plt.title("Эмпирическая функция распределения")
plt.xlabel('x, цены ананасов')
plt.ylabel('F(x), эмпирическое распределение')
plt.grid(True)
plt.show()

# Выводы
- В данных присутствуют выбросы.
- Распределение близко к симметричному.
- Нормальное распределение не описывает данные.
- Медиана ≈ среднему → симметрия.